Loading initial data

In [88]:
import pandas as pd

In [89]:
data_path = "data/cleaned_gas_monitoring.csv"
df = pd.read_csv(data_path)

In [90]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9735 entries, 0 to 9734
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Time of Day                9735 non-null   str    
 1   Temperature                9735 non-null   float64
 2   Humidity                   9735 non-null   float64
 3   CO2_InfraredSensor         9735 non-null   float64
 4   CO2_ElectroChemicalSensor  9735 non-null   float64
 5   MetalOxideSensor_Unit1     9735 non-null   float64
 6   MetalOxideSensor_Unit2     9735 non-null   float64
 7   MetalOxideSensor_Unit3     9735 non-null   float64
 8   MetalOxideSensor_Unit4     9735 non-null   float64
 9   CO_GasSensor               9735 non-null   float64
 10  Session ID                 9735 non-null   int64  
 11  HVAC Operation Mode        9735 non-null   str    
 12  Ambient Light Level        9735 non-null   str    
 13  Activity Level             9735 non-null   str    
dtypes: 

In [91]:
df.isnull().sum()

Time of Day                  0
Temperature                  0
Humidity                     0
CO2_InfraredSensor           0
CO2_ElectroChemicalSensor    0
MetalOxideSensor_Unit1       0
MetalOxideSensor_Unit2       0
MetalOxideSensor_Unit3       0
MetalOxideSensor_Unit4       0
CO_GasSensor                 0
Session ID                   0
HVAC Operation Mode          0
Ambient Light Level          0
Activity Level               0
dtype: int64

Missing Values, etc still remain so I'll be cleaning these to ensure its okay

In [92]:
df.duplicated().sum()

np.int64(57)

## Removing Duplicates first;

In [93]:
df_dropped = df.drop_duplicates()
df_dropped.info()
df_dropped.shape

<class 'pandas.DataFrame'>
Index: 9678 entries, 0 to 9734
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Time of Day                9678 non-null   str    
 1   Temperature                9678 non-null   float64
 2   Humidity                   9678 non-null   float64
 3   CO2_InfraredSensor         9678 non-null   float64
 4   CO2_ElectroChemicalSensor  9678 non-null   float64
 5   MetalOxideSensor_Unit1     9678 non-null   float64
 6   MetalOxideSensor_Unit2     9678 non-null   float64
 7   MetalOxideSensor_Unit3     9678 non-null   float64
 8   MetalOxideSensor_Unit4     9678 non-null   float64
 9   CO_GasSensor               9678 non-null   float64
 10  Session ID                 9678 non-null   int64  
 11  HVAC Operation Mode        9678 non-null   str    
 12  Ambient Light Level        9678 non-null   str    
 13  Activity Level             9678 non-null   str    
dtypes: float

(9678, 14)

## Identifying numerical and categorical cols

In [94]:
numerical_features = ['Temperature', 'Humidity', 'CO2_InfraredSensor', 'CO2_ElectroChemicalSensor', 'MetalOxideSensor_Unit1', 'MetalOxideSensor_Unit2', 'MetalOxideSensor_Unit3', 'MetalOxideSensor_Unit4', 'CO_GasSensor'] # session id not included as it is not a feature for modeling
categorical_features = ['Time of Day', 'HVAC Operation Mode', 'Ambient Light Level', 'Activity Level'] # activity level to remove later because its our target variable

## Handling null values
1. Numerical columns, we using Median 
2. Categorical column (ambient light level) by choosing the light level referencing the time of day for missing data. More realistic

In [95]:
for col in ['Humidity', 'MetalOxideSensor_Unit2', 'CO_GasSensor']:
    if col in numerical_features:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

if 'Ambient Light Level' in categorical_features:
    # Impute missing light levels using the mode of their specific Time of Day
    df['Ambient Light Level'] = df.groupby('Time of Day')['Ambient Light Level'].transform(
        lambda x: x.fillna(x.mode()[0])
    )

print("\nMissing values after imputation:")
print(df[numerical_features + categorical_features].isnull().sum())


Missing values after imputation:
Temperature                  0
Humidity                     0
CO2_InfraredSensor           0
CO2_ElectroChemicalSensor    0
MetalOxideSensor_Unit1       0
MetalOxideSensor_Unit2       0
MetalOxideSensor_Unit3       0
MetalOxideSensor_Unit4       0
CO_GasSensor                 0
Time of Day                  0
HVAC Operation Mode          0
Ambient Light Level          0
Activity Level               0
dtype: int64


## Train Test Split (before fit scaling/encoding)


In [96]:
from sklearn.model_selection import train_test_split

In [97]:
activity_mapping = {
    'Low Activity': 0,
    'Moderate Activity': 1,
    'High Activity': 2
}

y = df['Activity Level'].map(activity_mapping).to_numpy()

In [98]:
X_raw = df.drop(columns=['Activity Level', 'Session ID'])

In [99]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# Encoding

ASSUMPTION: Our problem statement ask us to find relationships. So I will encode the stuff with nominal encoding (no relationships/hierarchy), and assume nothing about them.

I will use One Hot Encoding on everything first, since it is a form of encoding that does not explictly say something has a relationship/hiererachy, etc)

But for activity level, i will add a hierarchy (because it is important)

In [100]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder

categorical_features = ['Time of Day', 'HVAC Operation Mode', 'Ambient Light Level']
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')
X_train_cat_encoded = encoder.fit_transform(X_train_raw[categorical_features])
X_test_cat_encoded = encoder.transform(X_test_raw[categorical_features])

encoded_feature_names = encoder.get_feature_names_out(categorical_features)

X_train_cat_df = pd.DataFrame(X_train_cat_encoded, columns=encoded_feature_names, index=X_train_raw.index)
X_test_cat_df = pd.DataFrame(X_test_cat_encoded, columns=encoded_feature_names, index=X_test_raw.index)


In [101]:
X_train_cat_df.head()

,Time of Day_evening,Time of Day_morning,Time of Day_night,HVAC Operation Mode_eco_mode,HVAC Operation Mode_heating_active,HVAC Operation Mode_maintenance_mode,HVAC Operation Mode_off,HVAC Operation Mode_ventilation_only,Ambient Light Level_dim,Ambient Light Level_moderate,Ambient Light Level_very_bright,Ambient Light Level_very_dim
4804,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
7662,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
469,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
9541,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
5071,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [102]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_raw[numerical_features])

X_test_num_scaled = scaler.transform(X_test_raw[numerical_features])

X_train_num_df = pd.DataFrame(X_train_num_scaled, columns=numerical_features, index=X_train_raw.index)
X_test_num_df = pd.DataFrame(X_test_num_scaled, columns=numerical_features, index=X_test_raw.index)

In [103]:
X_train_num_df.head()

,Temperature,Humidity,CO2_InfraredSensor,CO2_ElectroChemicalSensor,MetalOxideSensor_Unit1,MetalOxideSensor_Unit2,MetalOxideSensor_Unit3,MetalOxideSensor_Unit4,CO_GasSensor
4804,-0.539901,0.157235,1.039998,-0.181102,0.256503,-0.462294,-0.179337,-0.296703,-0.345882
7662,-0.289865,0.894869,-0.613356,1.423030,0.485657,-0.059308,0.245106,0.671489,-0.345882
469,1.569782,0.013727,0.201912,-1.621450,0.035109,-0.059308,-1.639308,-2.015952,-0.345882
9541,0.034401,0.217509,-0.423416,-1.213322,0.210312,0.675910,0.290042,0.622398,-0.345882
5071,-0.145312,0.846076,0.338349,-0.210399,0.239262,0.097976,0.359721,0.439865,-0.345882


In [104]:
X_train_final = pd.concat([X_train_num_df, X_train_cat_df], axis=1)
X_test_final = pd.concat([X_test_num_df, X_test_cat_df], axis=1)

In [105]:
X_train_final.head()

,Temperature,Humidity,CO2_InfraredSensor,CO2_ElectroChemicalSensor,MetalOxideSensor_Unit1,MetalOxideSensor_Unit2,MetalOxideSensor_Unit3,MetalOxideSensor_Unit4,CO_GasSensor,Time of Day_evening,...,Time of Day_night,HVAC Operation Mode_eco_mode,HVAC Operation Mode_heating_active,HVAC Operation Mode_maintenance_mode,HVAC Operation Mode_off,HVAC Operation Mode_ventilation_only,Ambient Light Level_dim,Ambient Light Level_moderate,Ambient Light Level_very_bright,Ambient Light Level_very_dim
4804,-0.539901,0.157235,1.039998,-0.181102,0.256503,-0.462294,-0.179337,-0.296703,-0.345882,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
7662,-0.289865,0.894869,-0.613356,1.423030,0.485657,-0.059308,0.245106,0.671489,-0.345882,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
469,1.569782,0.013727,0.201912,-1.621450,0.035109,-0.059308,-1.639308,-2.015952,-0.345882,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
9541,0.034401,0.217509,-0.423416,-1.213322,0.210312,0.675910,0.290042,0.622398,-0.345882,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
5071,-0.145312,0.846076,0.338349,-0.210399,0.239262,0.097976,0.359721,0.439865,-0.345882,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [106]:
X_test_final.head()

,Temperature,Humidity,CO2_InfraredSensor,CO2_ElectroChemicalSensor,MetalOxideSensor_Unit1,MetalOxideSensor_Unit2,MetalOxideSensor_Unit3,MetalOxideSensor_Unit4,CO_GasSensor,Time of Day_evening,...,Time of Day_night,HVAC Operation Mode_eco_mode,HVAC Operation Mode_heating_active,HVAC Operation Mode_maintenance_mode,HVAC Operation Mode_off,HVAC Operation Mode_ventilation_only,Ambient Light Level_dim,Ambient Light Level_moderate,Ambient Light Level_very_bright,Ambient Light Level_very_dim
6359,0.776697,-0.009235,0.171606,-0.531476,-0.939633,-1.205880,-0.685370,-0.837732,2.326467,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1445,-3.560073,0.157235,-0.143785,0.997785,-1.891251,-0.240047,-0.297703,-1.014751,3.662642,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1319,-0.227356,0.157235,-0.027421,0.768566,0.391354,-0.053102,0.234202,0.160761,0.990293,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4222,0.847020,0.183067,-0.108475,-1.122867,0.471981,-0.846058,-0.177998,-0.284298,0.990293,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
893,-0.414883,-1.857624,-1.385598,2.374195,2.355794,2.168760,1.277594,2.106205,-1.682057,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [107]:
print(y_train[:5])
print(y_test[:5])

[1 1 0 2 0]
[0 0 1 0 1]


# Cleaning Finished
# Moving to ML

In [108]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

In [109]:
# 1. Define your 3 baseline models and 3 parameterized versions
models = {
    # --- MODEL 1: LOGISTIC REGRESSION ---
    "Logistic Regression (Baseline)": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "Logistic Regression (Parameterized)": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        C=0.1,  # Stronger L2 regularization to prevent overfitting on synthetic noise
        solver="saga",  # Efficient solver for multi-class scaled data
        random_state=42,
    ),
    # --- MODEL 2: RANDOM FOREST ---
    "Random Forest (Baseline)": RandomForestClassifier(
        class_weight="balanced", random_state=42
    ),
    "Random Forest (Parameterized)": RandomForestClassifier(
        n_estimators=200,  # More trees for a smoother decision boundary
        max_depth=10,  # Limits depth to prevent memorizing contaminated data
        min_samples_split=5,  # Requires more evidence to split a node
        class_weight="balanced_subsample",  # Adjusts weights at each tree split
        random_state=42,
    ),
    # --- MODEL 3: XGBOOST ---
    "XGBoost (Baseline)": xgb.XGBClassifier(random_state=42),
    "XGBoost (Parameterized)": xgb.XGBClassifier(
        n_estimators=150,
        max_depth=5,  # Shallower trees prevent overfitting to sensor anomalies
        learning_rate=0.05,  # Slower learning rate prevents model from converging too fast on noise
        subsample=0.8,  # Trains each tree on 80% of rows to combat synthetic contamination
        colsample_bytree=0.8,  # Trains each tree on 80% of features
        random_state=42,
    ),
}

In [110]:
# 2. Store trained models for plotting importances later
trained_models = {}

print("training loop...\n")

for name, model in models.items():
    print(f"==================================================")
    print(f"⏳ Training: {name}...")

    # Train the model
    model.fit(X_train_final, y_train)
    trained_models[name] = model

    # Predict on the unseen validation test set
    y_pred = model.predict(X_test_final)

    # Print Evaluation Metrics
    print(f"Evaluation Report {name}:")
    # target_names maps numbers back to your human-readable activity strings
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=["Low Activity", "Moderate Activity", "High Activity"],
        )
    )

training loop...

⏳ Training: Logistic Regression (Baseline)...
Evaluation Report Logistic Regression (Baseline):
                   precision    recall  f1-score   support

     Low Activity       0.80      0.70      0.75      1122
Moderate Activity       0.57      0.49      0.53       612
    High Activity       0.20      0.43      0.28       213

         accuracy                           0.60      1947
        macro avg       0.53      0.54      0.52      1947
     weighted avg       0.66      0.60      0.63      1947

⏳ Training: Logistic Regression (Parameterized)...
Evaluation Report Logistic Regression (Parameterized):
                   precision    recall  f1-score   support

     Low Activity       0.80      0.70      0.75      1122
Moderate Activity       0.57      0.49      0.53       612
    High Activity       0.20      0.43      0.28       213

         accuracy                           0.60      1947
        macro avg       0.53      0.54      0.52      1947
     wei

# Feature Engineering

In [111]:
# Cell 1: Enable auto-reloading and import everything
%load_ext autoreload
%autoreload 2

from src.FeatureEngineering import load_cleaned_data, engineer_features

# Importing the feature engineering commands from Yao Hong

original_df = load_cleaned_data()
engineered_df = engineer_features(original_df)

# Let's see the glorious new feature columns your friend made
print(f"\nNew Dataframe Shape: {engineered_df.shape}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

Engineering features...
 -> Calculating temporal dynamics...
 -> Calculating rolling window statistics...
 -> Fusing metal oxide sensors...
 -> Calculating session baselines...
 -> Calculating Environmental Comfort, VOC Burden, and Discrepancy Flags...
 -> Encoding categorical variables...

New Dataframe Shape: (9735, 33)


In [119]:
# Split and Scaling the engineered DF

X = engineered_df.drop(columns=['Target_Activity', 'Session ID']) # both will not be used in training
y = engineered_df['Target_Activity'] # target variable for classification

# 2. Perform your stratified Train-Test Split to protect minority classes
X_train_raw_2, X_test_raw_2, y_train_2, y_test_2 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Identify columns that need scaling (Everything EXCEPT the true/false One-Hot columns)
# Booleans/Dummies shouldn't be scaled
cols_to_scale = [col for col in X.columns if not col.startswith(('HVAC', 'Time')) and X[col].nunique() > 2]

# Scaling strictly to protect against data leakage
scaler = StandardScaler()

# Copy dataframes to keep formatting intact
X_train_2 = X_train_raw_2.copy()
X_test_2 = X_test_raw_2.copy()

# Fit/Transform on train, ONLY transform on test
X_train_2[cols_to_scale] = scaler.fit_transform(X_train_raw_2[cols_to_scale])
X_test_2[cols_to_scale] = scaler.transform(X_test_raw_2[cols_to_scale])

print('Split')

Split


## Running Training Loop just to check

In [120]:
# All the 3 models
models = {
    "Logistic Regression (Baseline)": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Logistic Regression (Parameterized)": LogisticRegression(max_iter=1000, class_weight="balanced", C=0.1, solver="saga", random_state=42),
    
    "Random Forest (Baseline)": RandomForestClassifier(class_weight="balanced", random_state=42),
    "Random Forest (Parameterized)": RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_split=5, class_weight="balanced_subsample", random_state=42),
    
    "XGBoost (Baseline)": xgb.XGBClassifier(random_state=42),
    "XGBoost (Parameterized)": xgb.XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42)
}

# Run the training execution loop
for name, model in models.items():
    print(f"\n==================================================")
    print(f" Training: {name}...")
    model.fit(X_train_2, y_train_2)
    y_pred_2 = model.predict(X_test_2)
    
    print(f"Evaluation Report:")
    print(classification_report(y_test_2, y_pred_2, target_names=["Low Activity", "Moderate Activity", "High Activity"]))


 Training: Logistic Regression (Baseline)...
Evaluation Report:
                   precision    recall  f1-score   support

     Low Activity       0.83      0.67      0.74      1122
Moderate Activity       0.56      0.43      0.49       612
    High Activity       0.22      0.59      0.32       213

         accuracy                           0.59      1947
        macro avg       0.54      0.56      0.52      1947
     weighted avg       0.68      0.59      0.62      1947


 Training: Logistic Regression (Parameterized)...
Evaluation Report:
                   precision    recall  f1-score   support

     Low Activity       0.82      0.69      0.75      1122
Moderate Activity       0.55      0.45      0.50       612
    High Activity       0.22      0.52      0.31       213

         accuracy                           0.60      1947
        macro avg       0.53      0.55      0.52      1947
     weighted avg       0.67      0.60      0.62      1947


 Training: Random Forest (Baseli

Random Forest and XGBoost seems to be the models that are performing better than, and about the same. So I'll run more deeper hyperparameter tuning and enhancements to the model

# SMOTE

Initially in the EDA, we found out that our dataset are very imbalanced. Shown in the table at the end:


Activity Level distribution:
  Low Activity          :  5767  (57.7%)
  Moderate Activity     :  3138  (31.4%)
  High Activity         :  1095  (10.9%)

SMOTE creates synthetic data of the minority class to prevent models from becoming biased toward the majority class and avoids overfitting

In [114]:
# Install imbalanced-learn
from imblearn.over_sampling import SMOTE

# 1. Initialize SMOTE
smote = SMOTE(random_state=42)

# 2. Resample ONLY the training data to prevent data leakage
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_2, y_train_2)

print(f"Original training shape: {y_train_2.value_counts().to_dict()}")
print(f"Balanced training shape: {pd.Series(y_train_resampled).value_counts().to_dict()}")

Original training shape: {0: 4488, 1: 2448, 2: 852}
Balanced training shape: {1: 4488, 0: 4488, 2: 4488}


Testing the performance

In [116]:
smote_models = {
    "Logistic Regression (With SMOTE)": LogisticRegression(
        max_iter=1000, random_state=42
    ),
    "Random Forest (With SMOTE)": RandomForestClassifier(random_state=42),
    "XGBoost (With SMOTE)": xgb.XGBClassifier(random_state=42),
}

# removed class_weight adjustments because SMOTE has already balanced the classes, so we want the model to learn from the new distribution without further weighting

print("SMOTE Training Loop...\n")

# 2. Execute the loop using the resampled training data
for name, model in smote_models.items():
    print(f"==================================================")
    print(f"Training: {name} on balanced distribution...")

    # Fit ONCE on the SMOTE-resampled training data
    model.fit(X_train_resampled, y_train_resampled)

    # Predict ONCE on the untouched, realistic test set
    y_pred_3 = model.predict(X_test_2)

    # 3. Print out the validation dashboard for your report
    print(f"Evaluation Report for {name}:")
    print(
        classification_report(
            y_test_2,
            y_pred_3,
            target_names=["Low Activity", "Moderate Activity", "High Activity"],
        )
    )

    # Summary metrics for easy tracking (Macro f1 is best for imbalanced multi-class because it treats all classes equally0)
    macro_f1 = f1_score(y_test_2, y_pred_3, average="macro")
    print(f"Macro F1-Score on Unseen Test Set: {macro_f1:.4f}\n")

SMOTE Training Loop...

Training: Logistic Regression (With SMOTE) on balanced distribution...
Evaluation Report for Logistic Regression (With SMOTE):
                   precision    recall  f1-score   support

     Low Activity       0.76      0.70      0.73      1122
Moderate Activity       0.54      0.50      0.52       612
    High Activity       0.22      0.35      0.27       213

         accuracy                           0.60      1947
        macro avg       0.51      0.52      0.51      1947
     weighted avg       0.63      0.60      0.61      1947

Macro F1-Score on Unseen Test Set: 0.5051

Training: Random Forest (With SMOTE) on balanced distribution...
Evaluation Report for Random Forest (With SMOTE):
                   precision    recall  f1-score   support

     Low Activity       0.84      0.71      0.77      1122
Moderate Activity       0.52      0.62      0.57       612
    High Activity       0.29      0.36      0.32       213

         accuracy                    

# KV 5 fold cross validation Trial

# Grid Search/ Randomized CV